In [0]:
from pyspark.sql.functions import window,sum,desc,countDistinct,col
import pyspark.sql.functions as F

def gold_ingestion(gold_checkpoint_location):
    try:
        #Read from the silver Users_products_info_silver table.
        silver_df=spark.readStream.table("cosmetic_history.silver.users_products_info_silver")

        def aggregations(df, batch_id):
            
            # Create window start and end columns. 
            df= df.withColumn("window_start", col("window.start")) \
                .withColumn("window_end", col("window.end"))

            # Aggregating the data based on the window_start, window_end, category_code, event_type and brand.
            grouped_df=df.groupBy(col("window.start").alias("window_start"),col("window.end").alias("window_end"),"category_code","event_type","brand").agg(F.sum("price").alias("Total_Price"),countDistinct("product_id").alias("Total_No_Products"),countDistinct("user_id").alias("Total_No_Users"))

            grouped_df.createOrReplaceTempView('users_products_info_silver')
            
            merge_logic="""MERGE INTO cosmetic_history.gold.users_products_info_gold AS t
            USING users_products_info_silver AS s
            ON s.window_end=t.window_end and s.window_start=t.window_start AND s.category_code=t.category_code AND s.event_type=t.event_type AND s.brand=t.brand
            WHEN MATCHED
            THEN
            UPDATE
            SET t.Total_Price=s.Total_Price,t.Total_No_Products=s.Total_No_Products,t.Total_No_Users=s.Total_No_Users
            WHEN NOT MATCHED
            THEN
            INSERT (
            window_start,
            window_end,
            category_code,
            event_type,
            brand,
            Total_Price,
            Total_No_Products,
            Total_No_Users
            )
            VALUES (
            s.window_start,
            s.window_end,
            s.category_code,
            s.event_type,
            s.brand,
            s.Total_Price,
            s.Total_No_Products,
            s.Total_No_Users
            )"""
            spark.sql(merge_logic)
        
        # Writing the aggregated data to the gold table.

        gold_streaming_query=silver_df.writeStream.queryName("gold_streaming_query").option("checkpointLocation",gold_checkpoint_location)\
            .trigger(availableNow=True).foreachBatch(aggregations).outputMode('append').start()

        return gold_streaming_query
    except Exception as e:
        raise Exception(f"Error: {e}")

     
if __name__=="__main__":
    gold_checkpoint_location='/Volumes/cosmetic_history/checkpoint/cosmetic_history_checkpoints/gold_checkpoint'
    gold_streaming_query=gold_ingestion(gold_checkpoint_location)
    gold_streaming_query.awaitTermination()
    print('Done')
     

Done
